# 课后练习 · 电路分析初体验：亲手把感应电路找出来

> 对应课程第二章「机制可解释性入门：TransformerLens 与感应头」（ARENA 1.2）。课上讲过的感应电路，这次由你亲手把每一条证据重新拿一遍。

## 你在练习什么

**一句话**：语言模型会"从上文抄"。上文出现过 `Mr Dursley`，后面再看到 `Mr`，它就会预测 `Dursley`。这份练习用 13 张热力图，把这个能力一路追到**具体是哪几个头、它们怎么接在一起、算法怎么写在权重里**。走完一遍，就是一次完整的电路分析：

> **看到现象 → 提出假设 → 用越来越硬的证据验证 → 从权重里读出算法**

**第一幕 · 模型内部看得见吗？（第 1–2 题）**
模型的中间结果大多是几百维的向量，人读不懂。注意力模式是例外，可以直接画出来。你会看到 24 个头各有各的"看法"，其中有一个头的样子特别规整，先记住它。

**第二幕 · 发现现象，找出嫌疑人（第 3–6 题）**
你会亲手造一个只有"从上文抄"才能做对的任务，先确认模型真的会抄；再从 24 个头里找出负责"抄"的头，以及在它们上游负责"记住前一个 token"的头。到这一幕结束，你会拼出一个电路的**假设**，但手里只有"看起来像"的证据。

**第三幕 · 证据升级：从"看起来像"到"改了就变"（第 7–9 题）**
先拆账：正确答案的分数是哪些头加进去的？再动手：把一个头关掉，模型还能不能抄？最后在两条只差一个词的输入之间换掉一部分中间结果，看信息在模型里是怎么走的。这一幕有一个**反转**：两种方法会对同一个头给出相反的结论。想清楚为什么，你就理解了"归因"和"因果"的区别。

**第四幕 · 不给输入，只看权重（第 10–13 题）**
前面都是在具体的输入上观察。最后一幕把权重矩阵直接乘出来，读出每个头"看哪里""搬什么"，以及两个头是怎么接上的；再把上游的头剪断，检验这条连接在真实运行时是否成立。

**做完以后，希望你带走三件事**
1. 电路是一个**可以被验证的假设**，不是看着图编出来的故事
2. 证据有层级：**相关 → 归因 → 因果 → 权重**，只看其中一层可能会看错
3. 真正理解了一个电路，就能**预测**它在没见过的输入上会怎么做

## 怎么做

**每题只填一行代码。** 运行后会出现一张热力图，这张图就是这一题要你看懂的东西。每一题都按同样的节奏走：

1. 读 **想知道** 和 **先猜一猜**，先在心里给出一个猜测
2. 把标着 👉 的那一行里的 `...` 换成代码，然后运行。不会写就先点开 💡 提示；还是写不出来，再点开 🔑 答案
3. 看图，先自己用一句话说出"这张图在说什么"，再点开 📖 对答案

重点不在代码，在于看懂图。报错里如果出现 `ellipsis`，说明 `...` 还没换掉。

**环境**：推荐在 Google Colab 上做，先依次运行下面「0 · 准备」里的两格：
- **0.1 安装**：装 TransformerLens，约 1 分钟。torch、plotly 等 Colab 自带
- **0.2 加载模型**：第一次会从 HuggingFace 下载约 180 MB 的权重。如果看到 `HF_TOKEN` 相关的提示，可以忽略，这个模型是公开的，不需要登录

CPU 就能跑完全部 13 题。想快一点，可以在菜单 **Runtime（代码执行程序）→ Change runtime type（更改运行时类型）** 里选 **T4 GPU**。

---
## 0 · 准备

In [ ]:
# ===== 0.1 · 安装依赖：在 Colab 上先运行这一格（约 1 分钟）=====
# Colab 自带 torch、plotly、huggingface_hub，只缺 TransformerLens。
# 版本固定为本课验证过的 3.7.0。本地已经装好的话，可以跳过这一格。
%pip install -q transformer_lens==3.7.0

> 如果安装结束后 Colab 弹出 **Restart session（重新启动会话）**，点它，然后直接从下面的 0.2 继续运行，不需要重新安装。

In [ ]:
# ===== 0.2 · 加载模型和工具函数：运行这一格即可，不需要读懂 =====
import functools
import warnings
from pathlib import Path

import torch as t
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig

warnings.filterwarnings("ignore", category=DeprecationWarning)
t.set_grad_enabled(False)
device = "cuda" if t.cuda.is_available() else "cpu"

# ---------- 课上用的 2 层 attention-only 模型（ARENA 1.2 同款权重） ----------
cfg = HookedTransformerConfig(
    d_model=768, d_head=64, n_heads=12, n_layers=2, n_ctx=2048, d_vocab=50278,
    attention_dir="causal", attn_only=True, tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398, use_attn_result=True, normalization_type=None,
    positional_embedding_type="shortformer", device=device,
)
local = Path("attn_only_2L_half.pth")  # 放在同一目录就不用联网下载
weights = local if local.exists() else hf_hub_download("callummcdougall/attn_only_2L_half", "attn_only_2L_half.pth")
model = HookedTransformer(cfg)
model.load_state_dict(t.load(weights, map_location=device, weights_only=True))
print(f"模型已加载：{cfg.n_layers} 层 × {cfg.n_heads} 个注意力头，运行在 {device}")

# ---------- 画图 ----------
LAYERS = ["第 0 层", "第 1 层"]
HEADS = [str(h) for h in range(12)]
COMP_NAMES = ["嵌入"] + [f"{l}.{h}" for l in range(2) for h in range(12)]
SEQ = [[0, "#fcfcfb"], [0.15, "#cde2fb"], [0.35, "#86b6ef"], [0.6, "#3987e5"], [0.8, "#1c5cab"], [1, "#0d366b"]]
DIV = [[0, "#8a1c1c"], [0.25, "#e34948"], [0.45, "#f6c9c8"], [0.5, "#f0efec"],
       [0.55, "#b7d3f6"], [0.75, "#2a78d6"], [1, "#0d366b"]]  # 红 = 负，灰 = 0，蓝 = 正


def check(*xs):
    for x in xs:
        if x is Ellipsis:
            raise ValueError("👆 先把标着 👉 的那一行填好（把 ... 换成你的代码），再运行这一格")


def _np(z):
    return z.detach().float().cpu().numpy() if isinstance(z, t.Tensor) else z


def _uniq(labels):
    """同一个词出现多次时加不可见字符区分，免得坐标轴把它们合并成一格"""
    if labels is None or not isinstance(labels[0], str):
        return labels
    seen, out = {}, []
    for s in labels:
        seen[s] = seen.get(s, 0) + 1
        out.append(s + "​" * (seen[s] - 1))
    return out


def heatmap(z, title="", x=None, y=None, xlabel="", ylabel="", clabel="值", signed=False, text=False,
            zmin=0, zmax=None, height=500, vline=None):
    check(z)
    z = _np(z)
    if signed:
        m = zmax if zmax is not None else float(abs(z).max())
        scale = dict(color_continuous_scale=DIV, zmin=-m, zmax=m)
    else:
        scale = dict(color_continuous_scale=SEQ, zmin=zmin, zmax=zmax)
    x, y = _uniq(x), _uniq(y)
    fig = px.imshow(z, x=x, y=y, aspect="auto", text_auto=".2f" if text else False,
                    labels=dict(x=xlabel, y=ylabel, color=clabel), **scale)
    if x is not None and isinstance(x[0], str):
        fig.update_xaxes(tickmode="array", tickvals=x, tickangle=-60 if max(map(len, x)) > 3 else 0)
    if y is not None and isinstance(y[0], str):
        fig.update_yaxes(tickmode="array", tickvals=y)
    if vline is not None:
        fig.add_vline(x=vline, line_dash="dash", line_color="#52514e")
    fig.update_layout(title=title, height=height, template="plotly_white", margin=dict(t=60, b=40))
    fig.show()


def show_all_heads(cache, str_tokens):
    check(cache)
    pats = t.cat([cache["pattern", l][0] for l in range(2)])  # [24, 当前位置, 被看位置]
    fig = px.imshow(_np(pats), facet_col=0, facet_col_wrap=6, color_continuous_scale=SEQ, zmin=0, zmax=1,
                    facet_col_spacing=0.02, facet_row_spacing=0.06)
    names = [f"{l}.{h}" for l in range(2) for h in range(12)]
    fig.for_each_annotation(lambda a: a.update(text="头 " + names[int(a.text.split("=")[-1])]))
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_layout(title="24 个头的注意力模式（每个小图：行 = 当前位置，列 = 被看的位置）",
                      height=720, template="plotly_white", coloraxis_showscale=False)
    fig.show()


# ---------- 第 ② 节：重复的随机序列 ----------
def random_tokens(batch, seq_len):
    t.manual_seed(0)  # 固定随机种子，和课上的数字一致
    return t.randint(0, cfg.d_vocab, (batch, seq_len)).to(device)


def bos_column(batch):
    return t.full((batch, 1), model.tokenizer.bos_token_id).to(device)


def per_token_loss(logits, tokens):
    """每个位置预测下一个 token 的 loss，形状 [batch, 序列长度 - 1]"""
    return -logits.log_softmax(-1)[:, :-1].gather(-1, tokens[:, 1:, None])[..., 0]


def run_and_show_loss(rep_tokens, seq_len):
    check(rep_tokens)
    if tuple(rep_tokens.shape) != (10, 2 * seq_len + 1):
        raise ValueError(f"rep_tokens 的形状应该是 (10, {2 * seq_len + 1})，现在是 {tuple(rep_tokens.shape)}，检查一下 dim")
    logits, cache = model.run_with_cache(rep_tokens)
    loss = per_token_loss(logits, rep_tokens)
    heatmap(loss, title="每个位置的 loss（颜色越深 = 越猜不中）", x=list(range(1, 2 * seq_len + 1)),
            y=[f"序列 {i}" for i in range(10)], xlabel="被预测的 token 的位置", clabel="loss", height=380, vline=seq_len + 0.5)
    print(f"第一遍平均 loss = {loss[:, :seq_len].mean():.2f}    第二遍平均 loss = {loss[:, seq_len:].mean():.2f}")
    return logits, cache


def stack_patterns(cache):
    """24 个头的注意力，10 条序列取平均，形状 [2 层, 12 头, 当前位置, 被看位置]"""
    return t.stack([cache["pattern", l].mean(0) for l in range(2)])


def stripe_score(patterns, back):
    """取出注意力矩阵上"往回看 back 格"的那条斜线，求平均"""
    return patterns.diagonal(offset=-back, dim1=-2, dim2=-1).mean(-1)


# ---------- 第 ③ 节：归因、消融、patching ----------
def get_components(cache, tokens):
    """残差流 = 嵌入 + 24 个头的输出。返回这 25 个组件，以及每个位置上"正确答案"的方向 u"""
    comps = t.cat([cache["embed"][None],
                   cache["result", 0].permute(2, 0, 1, 3),
                   cache["result", 1].permute(2, 0, 1, 3)])[:, :, :-1]   # [25, batch, 位置, 768]
    answer_dirs = model.W_U[:, tokens[:, 1:]].permute(1, 2, 0)          # [batch, 位置, 768]
    return comps, answer_dirs


def show_dla(dla, logits, tokens, seq_len):
    check(dla)
    if tuple(dla.shape) != (25, *tokens[:, 1:].shape):
        raise ValueError(f"dla 的形状应该是 (25, 10, {tokens.shape[1] - 1})，现在是 {tuple(dla.shape)}")
    real = logits[:, :-1].gather(-1, tokens[:, 1:, None])[..., 0]
    err = (dla.sum(0) - real).abs().max().item()
    print(f"检验：25 项加起来 vs 模型真实的 logit，最大误差 {err:.1e}", "✅ 拆账成立" if err < 1e-2 else "❌ 对不上，检查一下")
    heatmap(dla.mean(1), signed=True, y=COMP_NAMES, x=list(range(1, 2 * seq_len + 1)), height=720,
            xlabel="被预测的 token 的位置", ylabel="组件", clabel="logit 贡献", vline=seq_len + 0.5,
            title="每个组件对正确答案 logit 的直接贡献（蓝 = 推高，红 = 压低）")
    first, second = dla[:, :, :seq_len].mean((1, 2)), dla[:, :, seq_len:].mean((1, 2))
    print(f"正确答案的平均 logit：第一遍 {real[:, :seq_len].mean():+.2f}  →  第二遍 {real[:, seq_len:].mean():+.2f}")
    print("两遍之间变化最大的组件（第一遍 → 第二遍）：")
    for i in (second - first).abs().argsort(descending=True)[:3].tolist():
        print(f"    {COMP_NAMES[i]:>4}: {first[i]:+.2f} → {second[i]:+.2f}")
    print(f"    {'0.7':>4}: {first[8]:+.2f} → {second[8]:+.2f}   ← 第 6 题找到的 0.7")


def ablation_scan(hook_fn, tokens, seq_len):
    """把 24 个头依次关掉，返回第二遍 loss 的上升量 [2, 12]"""
    def second_half_loss(logits):
        return per_token_loss(logits, tokens)[:, -(seq_len - 1):].mean()

    base = second_half_loss(model(tokens))
    scores = t.zeros(2, 12)
    for layer in range(2):
        for head in range(12):
            logits = model.run_with_hooks(tokens, fwd_hooks=[
                (f"blocks.{layer}.attn.hook_z", functools.partial(hook_fn, head=head))])
            scores[layer, head] = (second_half_loss(logits) - base).item()
    if scores.abs().max() < 1e-4:
        print("⚠️ 关掉哪个头都没有变化：hook 里那一行是不是还没填？")
    return scores


def make_tokens(words):
    return t.tensor([[model.tokenizer.bos_token_id] + [model.to_single_token(" " + w) for w in words]]).to(device)


PATCH_LAYERS = [("blocks.0.hook_resid_pre", "第 0 层输入（刚嵌入）"),
                ("blocks.1.hook_resid_pre", "第 1 层输入"),
                ("blocks.1.hook_resid_post", "第 1 层输出（最终）")]


def patching_scan(hook_fn, clean_tokens, corrupt_tokens, answer, wrong):
    """对每个（层, 位置）做一次 patching，返回恢复比例 [3, 位置]"""
    def logit_diff(logits):
        return (logits[0, -1, answer] - logits[0, -1, wrong]).item()

    clean_ld, corrupt_ld = logit_diff(model(clean_tokens)), logit_diff(model(corrupt_tokens))
    out = t.zeros(len(PATCH_LAYERS), clean_tokens.shape[1])
    for r, (name, _) in enumerate(PATCH_LAYERS):
        for pos in range(clean_tokens.shape[1]):
            logits = model.run_with_hooks(corrupt_tokens, fwd_hooks=[(name, functools.partial(hook_fn, pos=pos))])
            out[r, pos] = (logit_diff(logits) - corrupt_ld) / (clean_ld - corrupt_ld)
    if out.abs().max() < 1e-4:
        print("⚠️ 换哪一格都没有变化：hook 里那一行是不是还没填？")
    return out


# ---------- 第 ④ 节：只看权重 ----------
WORDS = ("apple river window garden pencil mirror winter bottle candle rocket orange tiger music "
         "island doctor silver planet coffee forest hammer rabbit castle butter jacket dragon").split()


def word_ids(words):
    return t.tensor([model.to_single_token(" " + w) for w in words]).to(device)


def copy_accuracy(W_OV, chunk=1000):
    """在全部 5 万个 token 上：每一行最大值落在对角线上（= 推高的正是它自己）的比例"""
    hits = 0
    for ids in t.arange(cfg.d_vocab, device=device).split(chunk):
        hits += ((model.W_E[ids] @ W_OV @ model.W_U).argmax(-1) == ids).sum().item()
    return hits / cfg.d_vocab


def center_rows(m):
    """每行减去这一行的平均值。softmax 只看同一行里谁高谁低，减掉平均值不改变结论，图更清楚"""
    check(m)
    return m - m.mean(1, keepdim=True)


def scores_to_pattern(scores):
    """注意力分数 → 注意力模式：缩放、只许看自己和前面、softmax"""
    check(scores)
    mask = t.tril(t.ones_like(scores)).bool()
    return t.where(mask, scores / cfg.d_head ** 0.5, -1e9).softmax(-1)


def run_cut(cut_fn, head):
    """剪断第 0 层的某个头，返回 1.4 在 10 条重复序列上的注意力 [10, 101, 101]"""
    store = {}

    def save(pattern, hook):
        store["p"] = pattern[:, 4].clone()

    model.run_with_hooks(rep_tokens, return_type=None, fwd_hooks=[
        ("blocks.0.attn.hook_v", functools.partial(cut_fn, head=head)),
        ("blocks.1.attn.hook_pattern", save)])
    return store["p"]


def compare_before_after(cut_fn, head=7):
    before, after = rep_cache["pattern", 1][0, 4], run_cut(cut_fn, head)[0]
    fig = make_subplots(1, 2, subplot_titles=["正常的 1.4", f"剪断 0.{head} 之后的 1.4"], horizontal_spacing=0.08)
    for i, z in enumerate([before, after]):
        fig.add_trace(go.Heatmap(z=_np(z), coloraxis="coloraxis"), 1, i + 1)
    fig.update_yaxes(autorange="reversed", title_text="当前位置", col=1)
    fig.update_yaxes(autorange="reversed", col=2)
    fig.update_xaxes(title_text="被看的位置")
    fig.update_layout(coloraxis=dict(colorscale=SEQ, cmin=0, cmax=1, colorbar_title_text="注意力"), height=460, template="plotly_white",
                      title="1.4 在重复序列上的注意力：剪断前 vs 剪断后")
    fig.show()


def targeted_scan(cut_fn):
    base = stripe_score(rep_cache["pattern", 1][:, 4].mean(0), seq_len - 1).item()
    scores = t.tensor([stripe_score(run_cut(cut_fn, h).mean(0), seq_len - 1).item() for h in range(12)])
    if (scores - base).abs().max() < 1e-4:
        print("⚠️ 剪哪个头都没有变化：hook 里那一行是不是还没填？")
    print(f"不剪的时候：1.4 的 induction score = {base:.2f}")
    heatmap(scores[None], x=[f"剪 0.{h}" for h in range(12)], y=["1.4 的 induction score"],
            clabel="分数", text=True, zmax=1, height=230, title="依次剪断第 0 层的每个头，1.4 还剩多少感应能力")


print("准备完成 ✅")

### 认识一下研究对象

- **2 层**，每层 **12 个注意力头**，没有 MLP。所以模型里所有"在位置之间搬信息"的计算，都在这 24 个头里
- 头的名字写成 **"层.头"**：`0.7` = 第 0 层第 7 个头，`1.4` = 第 1 层第 4 个头
- **怎么读注意力热力图**：**行 = 当前位置**（正在做预测的那个 token），**列 = 被看的位置**。每一行加起来等于 1，表示当前位置把注意力怎样分给前面各个位置。只能看自己和前面，所以右上角永远是空的
- 鼠标悬停在格子上，可以看到具体的数值

---
## ① 看见注意力

### 第 1 题 · 跑一次模型，把中间结果全存下来

**想知道**：模型内部有没有人能直接看懂的东西？

**先猜一猜**：24 个头"看"同一句话的方式会一样吗？

**做法**：用 `run_with_cache` 跑一次前向，模型每一步的中间结果（叫作**激活**）都会被存进 `cache`。下面的代码会把 24 个头的注意力模式全部画出来。

<details><summary>💡 提示</summary>

`model.run_with_cache(tokens)` 会返回两样东西：模型的输出 `logits`，和装着所有中间结果的 `cache`。
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
logits, cache = model.run_with_cache(tokens)
```
</details>

In [ ]:
text = ("Mr and Mrs Dursley, of number four, Privet Drive, were proud to say that "
        "they were perfectly normal, thank you very much. Mr and Mrs Dursley")
tokens = model.to_tokens(text)          # 句子 → token 编号，最前面自动加一个起始符 <|endoftext|>
str_tokens = model.to_str_tokens(text)  # 每个 token 的文字，后面当坐标轴用

# 👉 填这一行：跑一次模型，拿到输出 logits 和缓存 cache
logits, cache = ...

# ↓ 下面不用改
show_all_heads(cache, str_tokens)

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：模型内部"谁在看谁"可以直接画出来，而且不同的头分工完全不同。

**为什么**：注意力模式是 softmax 之后的概率，每一行就是"当前这个 token 把注意力分给了前面哪些 token"，人能直接读。看图大致能分出几类：

- **紧贴对角线下方的一条细线**：每个 token 都在看前一个 token（比如 `0.7`）
- **最左边一列很亮**：注意力都落在起始符上，相当于这个头在这里没事做
- **散开的一片**：同时在看很多位置

模型里大多数中间结果（每个位置上的几百维向量）没法直接读，注意力模式是少数能直接读的东西，所以电路分析常常从它开始。
</details>

### 第 2 题 · 把 0.7 单独拿出来看

**想知道**：上一题里那个画出一条细斜线的头 `0.7`，到底在看什么？

**先猜一猜**：如果它在"看前一个 token"，放大以后，每一行最亮的格子应该在哪？

**做法**：从 `cache` 里取出第 0 层、第 7 个头的注意力模式。

<details><summary>💡 提示</summary>

`cache["pattern", 层号]` 的形状是 `[batch, 头, 当前位置, 被看位置]`。这里只有 1 句话，batch 取第 0 条：`cache["pattern", 0][0, 7]`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
pattern = cache["pattern", 0][0, 7]
```
</details>

In [ ]:
# 👉 填这一行：取出第 0 层、第 7 个头的注意力模式，形状 [当前位置, 被看位置]
pattern = ...

# ↓ 下面不用改
heatmap(pattern, x=str_tokens, y=str_tokens, xlabel="被看的位置", ylabel="当前位置", clabel="注意力",
        zmax=1, height=700, title="头 0.7 的注意力模式")
print(f"每个 token 分给'前一个 token'的平均注意力：{pattern.diagonal(-1).mean():.2f}")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：`0.7` 是一个 **previous-token head**（前一个 token 头）：每个 token 几乎把全部注意力（平均 ≈ 0.97）放在紧挨着的前一个 token 上。

**为什么重要**：一个位置一开始只知道"我是谁"。`0.7` 把前一个 token 的信息搬过来以后，每个位置还知道了"我前面是谁"。记住这个能力，后面会看到它是感应电路的第一块零件。
</details>

---
## ② 找到感应头

**感应头（induction head）** 做的事可以写成一条规则：**`[A] [B] … [A]` → 预测 `[B]`**。当前 token 是 A 时，回头找上文里 A 上一次出现的位置，看它后面那个 B，把 B 抄过来当预测。比如上文出现过 "Mr Dursley"，后面再出现 "Mr"，就预测 "Dursley"。

为了把这种能力单独拎出来，我们用**重复的随机序列**：随机抽 50 个 token，再原样重复一遍。随机 token 之间没有任何搭配可背，所以第二遍能不能猜对，只取决于模型会不会"从上文抄"。

### 第 3 题 · 造一段"重复的随机序列"

**想知道**：这个模型会不会"从上文抄"？

**先猜一猜**：第一遍是纯随机，loss 应该很高。第二遍呢？如果模型会抄，第二遍的 loss 会怎样？如果不会呢？

**做法**：把 50 个随机 token 原样拼两遍，最前面放一个起始符，一共造 10 条这样的序列。下面的代码会算出每个位置的 **loss**（预测下一个 token 时有多"意外"，越低 = 猜得越准）并画出来。

<details><summary>💡 提示</summary>

`t.cat([a, b, c], dim=1)` 把几段沿"序列"方向（第 1 维）首尾拼起来。这里三段分别是 `bos`、`half`、`half`。
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
rep_tokens = t.cat([bos, half, half], dim=1)
```
</details>

In [ ]:
seq_len = 50
half = random_tokens(batch=10, seq_len=seq_len)   # 10 条 × 50 个随机 token
bos = bos_column(batch=10)                         # 每条最前面的起始符

# 👉 填这一行：拼成 [起始符, 随机的 50 个, 同样的 50 个]，形状 [10, 101]
rep_tokens = ...

# ↓ 下面不用改（跑模型、存 cache、画每个位置的 loss）
rep_logits, rep_cache = run_and_show_loss(rep_tokens, seq_len)

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：虚线左边（第一遍）一片深色，右边（第二遍）一下子变浅，平均 loss 从 ≈ 14.5 断崖掉到 ≈ 4.3。

**为什么**：随机 token 之间没有任何规律，第二遍能猜对的唯一办法，是在上文找到当前 token 上次出现的地方，把它后面那个 token 抄过来。所以这张图说明模型里**存在**一个复制机制。

细节：第二遍的第一格（位置 51）仍然是深色，因为预测它时手里只有第一遍的最后一个 token，没有可抄的。
</details>

### 第 4 题 · 谁在抄？看 1.4 的注意力

**想知道**：复制是哪个头在做？它在看哪里？

**先猜一猜**：如果 `1.4` 是感应头，第二遍的某个位置应该看第一遍的哪个位置？（要找"同一个 token 上次出现的位置的**下一个**"。）在热力图上会画出什么形状？

**做法**：取出第 1 层、第 4 个头在第 0 条序列上的注意力模式。

<details><summary>💡 提示</summary>

和第 2 题一样，只是换成重复序列的缓存 `rep_cache`、第 1 层、第 4 个头：`rep_cache["pattern", 1][0, 4]`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
pattern = rep_cache["pattern", 1][0, 4]
```
</details>

In [ ]:
# 👉 填这一行：取出第 1 层、第 4 个头、第 0 条序列的注意力模式，形状 [101, 101]
pattern = ...

# ↓ 下面不用改
heatmap(pattern, xlabel="被看的位置", ylabel="当前位置", clabel="注意力", zmax=1, height=650,
        title="头 1.4 在重复序列上的注意力")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：左下那块（第二遍看第一遍）有一条斜线，离主对角线正好 49 格。第二遍的每个位置，都在看"同一个 token 上次出现的位置再往后一格"。这就是感应头的标志。

**为什么是 49 格**：第二遍位置 p 上的 token，就是第一遍位置 p − 50 上的 token。它上次出现时后面跟着的 token 在 p − 50 + 1 = p − 49。看那里、把那个 token 抄过来，就是正确答案。

**第一遍呢**：第一遍没有可抄的，注意力都落在最左一列的起始符上，相当于"没事做"。
</details>

### 第 5 题 · 用一个数给 24 个头打分

**想知道**：除了 `1.4`，还有没有别的感应头？24 张图一张张看太累了。

**先猜一猜**：感应头会在第几层？（感应要先知道"我前面是谁"，第 0 层的头能做到吗？）

**做法**：把第 4 题的斜线变成一个数：取每个头注意力矩阵上"往回看 `back` 格"那条斜线的平均值，叫 **induction score**。工具 `stripe_score(all_patterns, back)` 已经写好，你只需要决定 `back` 是多少。

<details><summary>💡 提示</summary>

`back` 就是第 4 题那条斜线离主对角线的距离，用 `seq_len` 来写：`stripe_score(all_patterns, back=seq_len - 1)`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
induction_score = stripe_score(all_patterns, back=seq_len - 1)
```
</details>

In [ ]:
all_patterns = stack_patterns(rep_cache)   # 24 个头的注意力（10 条序列平均），形状 [2, 12, 101, 101]

# 👉 填这一行：感应头往回看几格？算出每个头的 induction score，形状 [2, 12]
induction_score = ...

# ↓ 下面不用改
heatmap(induction_score, x=HEADS, y=LAYERS, xlabel="头", clabel="分数", text=True, zmax=1, height=260,
        title="Induction score：往回看 seq_len − 1 格的平均注意力")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：24 个头里只有 `1.4`（0.67）和 `1.10`（0.85）亮，其余都低于 0.05。感应头就这两个，都在第 1 层。

**为什么在第 1 层**：要找"前面是 A"的位置，被看的那个位置得先带着"我前面是谁"的信息。第 0 层的头读到的只有原始 token，还没有任何头把"前一个是谁"搬过来，所以它们做不了感应。

**方法上**：先看图（第 4 题），再写统计量（本题），统计量要和图互相印证。检测器没有"客观正确"的写法，阈值和斜线位置都是人定的。
</details>

### 第 6 题 · 同一把尺子换个刻度：找 previous-token head

**想知道**：第 1 层的感应头需要"我前面是谁"，这条信息是谁搬过来的？

**先猜一猜**：这样的头应该往回看几格？会在哪一层？

**做法**：同一个工具，只改 `back`。

<details><summary>💡 提示</summary>

"看前一个"就是往回看 1 格：`stripe_score(all_patterns, back=1)`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
prev_score = stripe_score(all_patterns, back=1)
```
</details>

In [ ]:
# 👉 填这一行：同样的工具，只改 back，算出每个头"看前一个"的分数，形状 [2, 12]
prev_score = ...

# ↓ 下面不用改
heatmap(prev_score, x=HEADS, y=LAYERS, xlabel="头", clabel="分数", text=True, zmax=1, height=260,
        title="Previous-token score：往回看 1 格的平均注意力")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：只有 `0.7` 一个头亮（0.99），其余都低于 0.3。它是全模型唯一的 previous-token head，在第 0 层，正好在感应头的上游。

到这里，我们拼出了一个**假设**：

```
0.7（第 0 层）     ：每个位置写下"我前面是谁"
      ↓ 1.4 / 1.10 在找位置时，读的就是这条信息
1.4 / 1.10（第 1 层）：当前是 A → 找"前面是 A"的位置 → 那里是 B → 把 B 抄过来
```

但这还只是**相关**证据：我们看到 `0.7` 在"看前一个"，并不知道 `1.4` 真的在用它。下一节开始拿更硬的证据。
</details>

---
## ③ 归因与消融

前 6 题都是"看"，得到的是**相关**证据：现象存在，但可能只是副产品。证据还有两个更硬的层级：

| 层级 | 问的问题 | 本节 |
|---|---|---|
| 相关 | 它看起来在做这件事吗？ | 第 1–6 题 |
| 归因 | 输出的分数里，有多少是它直接贡献的？ | 第 7 题 |
| 因果 | 改掉它，输出会跟着变吗？ | 第 8–9 题 |

改模型内部要用 **hook**：在前向计算到某个中间结果时，调用一个你写的小函数，这个函数可以读它，也可以改它。

### 第 7 题 · 拆账：正确答案的分数是谁加进去的（DLA）

先补一个事实。每个位置上有一条 768 维的向量，叫**残差流**：一开始是 token 的嵌入，之后每个头都把自己的输出**加**到上面。所以

> 最后的残差流 = 嵌入 + 24 个头的输出（一共 25 项）

模型给正确答案打的分（**logit**）= 最后的残差流 · **u**（u 是正确答案对应的那个方向）。点积可以对加法拆开：

> logit = 嵌入 · u + 头 0.0 · u + … + 头 1.11 · u

每一项就是一个组件对正确答案的**直接贡献**。这个方法叫 **Direct Logit Attribution（DLA）**。

**想知道**：第二遍里，正确答案的分数是谁加进去的？`0.7` 加了多少？

**先猜一猜**：如果上一题的假设对，哪两行会亮？`0.7` 那一行呢？

**做法**：`components` 是 25 个组件在每个位置的输出，`answer_dirs` 是每个位置上正确答案的方向 u。对每个组件求它和 u 的点积。

<details><summary>💡 提示</summary>

点积 = 逐元素相乘，再把最后一维（768）加起来。形状 `[25, 10, 100, 768]` 和 `[10, 100, 768]` 可以直接相乘（自动对齐后面三维）：`(components * answer_dirs).sum(-1)`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
dla = (components * answer_dirs).sum(-1)
```
</details>

In [ ]:
components, answer_dirs = get_components(rep_cache, rep_tokens)   # [25, 10, 100, 768]、[10, 100, 768]

# 👉 填这一行：每个组件的输出和正确答案方向 u 做点积，形状 [25, 10, 100]
dla = ...

# ↓ 下面不用改（先检验 25 项之和 = 真实 logit，再画 10 条序列的平均）
show_dla(dla, rep_logits, rep_tokens, seq_len)

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：虚线右边只有两行亮：`1.10`（≈ +6.3）和 `1.4`（≈ +2.7）。正确答案的平均 logit 从第一遍的 ≈ −5.0 涨到第二遍的 ≈ +5.8，多出来的约 11 分几乎都是这两个头写进去的。`0.7` 那一行几乎是 0。

**为什么能这样拆**：这个模型没有 LayerNorm，残差流是一路"加"出来的，所以分数可以精确地按组件拆开（代码里检验了 25 项之和等于真实 logit）。

**盲区**：DLA 只记"直接写进答案"的账。`0.7` 的输出先被 `1.4` 读走，功劳记在了 `1.4` 名下。所以 `0.7` 这一行是 0，**不代表它没用**。它是不是必需的，要动手把它关掉才知道。
</details>

### 第 8 题 · 消融：关掉一个头，看 loss 涨多少

**想知道**：`0.7` 是不是必需的？

**先猜一猜**：如果 `0.7` 只是一个"看起来在看前一个"的副产品，关掉它，第二遍的 loss 会怎样？如果它是电路的上游呢？

**做法**：**消融（ablation）** 就是把一个组件关掉再看输出。这里用 hook 在每个头的输出 `hook_z`（这个头搬来的内容，还没写回残差流）上，把指定的头置零，看第二遍的 loss 涨了多少。24 个头各做一次。

<details><summary>💡 提示</summary>

`z` 的四个维度依次是 `[batch, 位置, 头, 64]`。把第 `head` 个头在所有 batch、所有位置上的输出全部置零：`z[:, :, head, :] = 0.0`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
z[:, :, head, :] = 0.0
```
</details>

In [ ]:
def zero_ablate(z, hook, head):
    # z：某一层所有头的输出，形状 [batch, 位置, 头, 64]
    # 👉 填这一行：把第 head 个头的输出全部置零
    ...


# ↓ 下面不用改（24 次前向，CPU 上大约半分钟）
loss_increase = ablation_scan(zero_ablate, rep_tokens, seq_len)
heatmap(loss_increase, x=HEADS, y=LAYERS, xlabel="头", clabel="loss 变化", signed=True, text=True, height=260,
        title="关掉这个头后，第二遍 loss 变化了多少（蓝 = 上升，红 = 下降）")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：最亮的是 `0.7`（loss +9.4），比两个感应头 `1.10`（+5.8）、`1.4`（+2.6）都大。`0.7` 在 DLA 里几乎是 0，关掉它却让感应彻底失效。

**为什么**：`0.7` 不直接写答案，但它提供"我前面是谁"，没有它，`1.4` 和 `1.10` 就找不到该看的位置。这是第一份**因果**证据：改了内部，输出跟着变。

**意外收获**：`0.4`（+2.9）和 `0.11`（+1.6）也亮了，前面看注意力时它们并不显眼。消融常常会找出你没预料到的组件。

**红色的格子**：关掉 `0.3`，loss 反而下降 1.75。回看第 7 题，`0.3` 那一行从头到尾都是红的：它一直在压低正确答案，关掉这个"帮倒忙"的头，模型反而猜得更准。

**怎么读消融结果**：
- 影响大 = **必要**；影响小 ≠ 没用，可能有备份（`1.4` 和 `1.10` 就互为备份，关掉一个，另一个还能顶一部分）
- 置零是模型训练时从没见过的值，结果里会混进一点"分布外"的副作用。更讲究的做法是换成平均值，或换成另一条输入上的值，后者就是下一题
</details>

### 第 9 题 · Activation patching：区分两条输入的信息在哪里

消融是把激活"关掉"。**Activation patching** 更进一步：准备两条**只差一个词**的输入，在 corrupted（被改过的那条）的前向中，把某一层、某一位置的激活换成 clean（原始那条）上的值，看答案能恢复多少。

- clean：`apple river window garden pencil mirror … | apple river window garden` → 应该预测 **pencil**
- corrupted：第一遍的 pencil 换成 **rocket** → 应该预测 **rocket**
- 度量：logit(pencil) − logit(rocket)。**恢复比例** = 1 表示这一格携带了区分两条输入的全部信息，0 表示无关

**想知道**：区分 pencil 和 rocket 的那份信息，在每一层分别待在哪个位置？什么时候被搬到最后一个位置？

**先猜一猜**：在第 0 层输入（刚嵌入完），哪一格应该是 1？在最终输出，哪一格是 1？中间那一层呢？

**做法**：hook 把 corrupted 上某一格的残差流换成 clean 上的值。对 3 个层 × 15 个位置各做一次。

<details><summary>💡 提示</summary>

`resid` 的形状是 `[1, 位置, 768]`。`hook.name` 是当前这一层的名字，所以 clean 上同一层的残差流就是 `clean_cache[hook.name]`。只换第 `pos` 个位置：`resid[:, pos] = clean_cache[hook.name][:, pos]`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
resid[:, pos] = clean_cache[hook.name][:, pos]
```
</details>

In [ ]:
words = ["apple", "river", "window", "garden", "pencil", "mirror", "winter", "bottle", "candle", "dragon"]
clean_tokens = make_tokens(words + words[:4])               # 第二遍停在 garden，下一个应该是 pencil
corrupt_tokens = clean_tokens.clone()
corrupt_tokens[0, 5] = model.to_single_token(" rocket")     # 第一遍的 pencil 换成 rocket
clean_logits, clean_cache = model.run_with_cache(clean_tokens)


def patch_one_position(resid, hook, pos):
    # resid：corrupted 输入上这一层的残差流，形状 [1, 位置, 768]
    # 👉 填这一行：把第 pos 个位置换成 clean 输入上同一层、同一位置的值
    ...


# ↓ 下面不用改（3 层 × 15 个位置 = 45 次前向）
pencil, rocket = model.to_single_token(" pencil"), model.to_single_token(" rocket")
recovery = patching_scan(patch_one_position, clean_tokens, corrupt_tokens, answer=pencil, wrong=rocket)
labels = model.to_str_tokens(clean_tokens)
labels[5] = " pencil/rocket"
heatmap(recovery, x=labels, y=[name for _, name in PATCH_LAYERS], xlabel="位置", clabel="恢复比例", text=True, zmax=1,
        height=330, title="把这一格换成 clean 的值，答案恢复了多少")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：区分 pencil/rocket 的信息，一开始待在它自己的位置；到第 1 层输入时还在那里（≈ 0.9）；到第 1 层输出时，已经整份跑到了最后一个位置（1.0）。

**为什么**：信息从 pencil 那一格直接跳到最后一格，中间没有经过任何别的位置，这是第 1 层的注意力做的一次"直接搬运"。搬运的正是感应头：它在最后的 garden 上找"前面是 garden"的位置（就是 pencil 那一格），把那里的 token 抄过来。第 1 层输入那一格只有 0.9，剩下的一点被 `0.7` 抄到了下一个位置 mirror 上。

**和消融的区别**：消融问"这个组件是不是必需的"；patching 问"区分两条输入的信息在哪里、怎么移动"。
</details>

---
## ④ 从权重读出电路

前三节都是在具体输入上观察、干预。这一节**不给任何输入，只看权重矩阵**：如果"看哪里"和"搬什么"真的写在权重里，应该能直接读出来。

一个注意力头做两件事，各对应一个矩阵：

- **QK 电路**（看哪里）：`W_Q W_Kᵀ`，决定当前位置（query）和被看位置（key）之间的匹配分数
- **OV 电路**（搬什么）：`W_V W_O`，决定被看到的内容以什么样子写回残差流

残差流的 768 维没有人能读懂的含义，所以把矩阵两边乘上能读的东西：`W_E`（token → 向量）、`W_U`（向量 → 每个 token 的分数）、`W_pos`（位置 → 向量）。乘完以后，表格的行和列就都是 token 或位置了。

### 第 10 题 · 1.4 的 OV 电路：它搬什么

**想知道**：`1.4` 注意到一个 token 之后，会把什么写进预测？

**先猜一猜**：如果它在"复制"，注意到 apple 时应该推高哪个输出 token？25 × 25 的表上会是什么形状？

**做法**：取 25 个英文单词，算 `W_E → OV(1.4) → W_U` 这条链。第 X 行第 Y 列 = 注意到 X 时，给输出 Y 加多少分。

<details><summary>💡 提示</summary>

三个矩阵依次相乘，`W_E` 取这 25 个词的行，`W_U` 取这 25 个词的列：`model.W_E[ids] @ W_OV_14 @ model.W_U[:, ids]`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
full_OV = model.W_E[ids] @ W_OV_14 @ model.W_U[:, ids]
```
</details>

In [ ]:
ids = word_ids(WORDS)                              # 25 个单词的 token 编号
W_OV_14 = model.W_V[1, 4] @ model.W_O[1, 4]        # 1.4 的 OV 矩阵，[768, 768]

# 👉 填这一行：完整 OV 电路 = 词嵌入 → OV → 反嵌入，只取这 25 个词，形状 [25, 25]
full_OV = ...

# ↓ 下面不用改（softmax 只看同一行里谁高谁低，所以每行减去平均值再画，图更清楚）
heatmap(center_rows(full_OV), x=WORDS, y=WORDS, xlabel="输出 token（被推高的）",
        ylabel="注意到的 token", clabel="logit（减去行均值）", signed=True, height=650, title="1.4 的完整 OV 电路：注意到 X，给输出 Y 加多少分")
W_OV_110 = model.W_V[1, 10] @ model.W_O[1, 10]
print("在全部 5 万个 token 上，'推得最高的正是它自己'的比例：")
for name, W in [("1.4", W_OV_14), ("1.10", W_OV_110), ("1.4 + 1.10", W_OV_14 + W_OV_110)]:
    print(f"    {name:>10}: {copy_accuracy(W):.1%}")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：一条亮的对角线：注意到哪个词，就推高哪个词。`1.4` 的 OV 电路是一台**复制机**。这就是感应规则里"把 B 抄过来"那一步。

**数字**：在全部 5 万个 token 上，`1.4` 单独 30.8%，`1.10` 单独 76.8%，两个头合起来 95.6%。随机的话只有 0.002%。

**为什么两个头才够**：每个头的 OV 都要先把 768 维压到 64 维再展开回去，一个头能表达的东西有限。要对 5 万个不同的 token 都做到"原样复制"，一个头不够，两个头合力才接近完美。

**试一试**：把 `[1, 4]` 换成 `[1, 10]`，对角线会更干净。
</details>

### 第 11 题 · 0.7 的 QK 电路：它看哪里

**想知道**：`0.7` "看前一个"，是由 token 的内容决定的，还是只由位置决定？

**先猜一猜**：如果只由位置决定，只把**位置**代入 `0.7` 的 QK 电路，算出来的注意力应该是什么形状？

**做法**：这个模型的位置信息只进 Q 和 K，不进残差流，所以可以把位置单独拿出来：`W_pos · W_QK · W_posᵀ` 就是"位置 i 给位置 j 打的分"。下面的代码再做缩放、遮掉未来、softmax，变成注意力模式。

<details><summary>💡 提示</summary>

`model.W_pos` 的形状是 `[2048, 768]`（每个位置一行）：`model.W_pos @ W_QK_07 @ model.W_pos.T`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
pos_scores = model.W_pos @ W_QK_07 @ model.W_pos.T
```
</details>

In [ ]:
W_QK_07 = model.W_Q[0, 7] @ model.W_K[0, 7].T     # 0.7 的 QK 矩阵，[768, 768]

# 👉 填这一行：位置 × 位置的打分表 = 位置嵌入 · W_QK · 位置嵌入的转置，形状 [2048, 2048]
pos_scores = ...

# ↓ 下面不用改
pos_pattern = scores_to_pattern(pos_scores)
heatmap(pos_pattern[:20, :20], x=list(range(20)), y=list(range(20)), xlabel="被看的位置", ylabel="当前位置",
        clabel="注意力", zmax=1, height=560, title="只用位置算出的 0.7 注意力（前 20 个位置）")
print(f"全部 2048 个位置上，分给'前一个位置'的平均注意力：{pos_pattern.diagonal(-1).mean():.3f}")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：只用位置，`0.7` 就已经把几乎全部注意力（平均 0.998）放在前一格上。"看前一个"直接写在权重里，跟 token 的内容无关。

**为什么这比第 2 题更强**：第 2 题只在一句话上看到它这样做；这里没有用任何输入，说明它对**所有**输入、所有位置都这样做。

（第 0 行只能看它自己，因为第 0 个位置前面什么都没有。）
</details>

### 第 12 题 · 两个头怎么接上：K-composition

后面的头可以从三个入口读取前面的头写进残差流的东西：query、key、value。感应头用的是 **K-composition**：`1.4` 算 **key** 时，读的是 `0.7` 写进来的"我前面是谁"。于是 key 表达的不再是"我是谁"，而是"我前面是谁"。

**想知道**：在权重层面，`1.4` 的 query 和"经 `0.7` 搬运后的 key"是怎么匹配的？

**先猜一猜**：感应头要找"前一个 token = 当前 token"的位置。query 是"当前是 A"、key 是"我前面是 X"，A 和 X 是什么关系时分数最高？表上是什么形状？

**做法**：
- query 一侧：当前 token A，直接经 `W_E`、`W_Q(1.4)`
- key 一侧：前一个 token X，先经 `W_E`，再被 `0.7` 的 OV 搬过来，再经 `W_K(1.4)`

两侧做点积，得到 25 × 25 的匹配分数。

<details><summary>💡 提示</summary>

沿着 key 一侧的路径依次相乘：`model.W_E[ids] @ W_OV_07 @ model.W_K[1, 4]`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
k_side = model.W_E[ids] @ W_OV_07 @ model.W_K[1, 4]
```
</details>

In [ ]:
q_side = model.W_E[ids] @ model.W_Q[1, 4]          # 当前 token A → 1.4 的 query，[25, 64]
W_OV_07 = model.W_V[0, 7] @ model.W_O[0, 7]        # 0.7 的 OV 矩阵，[768, 768]

# 👉 填这一行：前一个 token X → 被 0.7 搬过来 → 1.4 的 key，形状 [25, 64]
k_side = ...

# ↓ 下面不用改（和第 10 题一样，每行减去平均值再画）
check(k_side)
kcomp = q_side @ k_side.T                          # [25, 25]：行 = 当前 token A，列 = 前一个 token X
heatmap(center_rows(kcomp), x=WORDS, y=WORDS, xlabel="key：前一个 token X（经 0.7 搬运）",
        ylabel="query：当前 token A", clabel="注意力分数（减去行均值）", signed=True, height=650, title="1.4 的 query × 经 0.7 搬运的 key")
print(f"每一行分数最高的格子落在对角线上的比例：{(kcomp.argmax(1).cpu() == t.arange(len(WORDS))).float().mean():.0%}")

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：又是一条对角线。当前 token 是 A 时，`1.4` 最想看的是"前一个 token 也是 A"的位置。这正是感应规则里"找到上次出现的 A，看它后面那个"的那一步，而且它是由 `0.7` 和 `1.4` 两个头的权重**乘在一起**才成立的。

**为什么非要 K-composition**：如果 `1.4` 的 key 只读原始 token，它只能找到"自己就是 A"的位置，看到的是 A 本身，而不是 A 后面的 B。有了 `0.7` 写进来的"我前面是谁"，它才能落在 B 上，再由第 10 题的 OV 把 B 抄过来。

**试一试**：把 `W_OV_07` 换成第 0 层别的头（比如 `model.W_V[0, 2] @ model.W_O[0, 2]`），对角线会消失。只有 `0.7` 能接上。
</details>

### 第 13 题 · 定向消融：剪断 0.7，1.4 还看得见吗

**想知道**：第 12 题从权重上读出 `1.4` 的 key 依赖 `0.7`。在真实的前向里把 `0.7` 剪断，`1.4` 还能找到该看的位置吗？

**先猜一猜**：如果这条连接是真的，第 4 题那条斜线会怎样？注意力会跑到哪里去？如果剪断后斜线还在，又说明什么？

**做法**：在第 0 层的 value（`hook_v`：每个头准备搬运的内容）上把 `0.7` 置零。这样 `0.7` 照样"看前一个"，但什么也搬不过去。然后不看 loss，而是看下游 `1.4` 的注意力。

<details><summary>💡 提示</summary>

`v` 的形状是 `[batch, 位置, 头, 64]`，和第 8 题的 `z` 一样：`v[:, :, head, :] = 0.0`
</details>

<details><summary>🔑 答案（先自己试，实在写不出来再点开）</summary>

```python
v[:, :, head, :] = 0.0
```
</details>

In [ ]:
def cut_head(v, hook, head):
    # v：第 0 层所有头准备搬运的内容，形状 [batch, 位置, 头, 64]
    # 👉 填这一行：把第 head 个头要搬运的内容全部置零
    ...


# ↓ 下面不用改
compare_before_after(cut_head, head=7)   # 左：正常的 1.4；右：剪断 0.7 之后的 1.4
targeted_scan(cut_head)                  # 依次剪断第 0 层每个头，看 1.4 的 induction score

<details><summary>📖 看完图再点开：这张图在说什么</summary>

**核心意思**：剪断 `0.7`，`1.4` 的斜线消失了，注意力全部退回最左边的起始符，induction score 从 0.67 掉到 0.02。剪断第 0 层别的头，斜线基本还在（0.60–0.86）。`0.4` 是部分例外（→ 0.43），和第 8 题里它也亮了一致。

**为什么**：这一行代码和第 8 题几乎一样，干预手法没变，变的是**看哪里**。第 8 题看最终的 loss，问"谁是必需的"；这里看下游某个头的注意力，问"谁和谁连着"。它和第 12 题从权重读出的连接互相印证：一个来自权重，一个来自真实的前向。
</details>

---
## 收尾：你亲手拿到的整条证据链

```
W_E / W_pos ──► 0.7  previous-token head  ──K-composition──►  1.4 / 1.10  induction heads ──► W_U
               "我前面是谁"                                   "找前面是 A 的位置，把那里的 B 抄过来"
```

| 零件 | 证据 | 层级 |
|---|---|---|
| `0.7` 看前一个 | 第 2、6 题：看前一个 ≈ 0.99 | 相关 |
| | 第 8 题：关掉它 loss +9.4，全模型最大 | 因果 |
| | 第 11 题：只靠位置就看前一格，0.998 | 权重 |
| `0.7 → 1.4` 的连接 | 第 12 题：两个头的权重相乘是一条对角线 | 权重 |
| | 第 13 题：剪断 0.7，1.4 的 induction score 0.67 → 0.02 | 因果 |
| `1.4 / 1.10` 复制 | 第 4、5 题：斜线，induction score 0.67 / 0.85 | 相关 |
| | 第 7 题：第二遍多出的 ~11 分几乎都是它们写的 | 归因 |
| | 第 8 题：关掉它们 loss +2.6 / +5.8 | 因果 |
| | 第 9 题：信息在第 1 层被搬到最后一个位置 | 因果 |
| | 第 10 题：OV 是复制机，两个头合起来 95.6% | 权重 |

**电路分析就是这样一个循环**：从一个现象出发（第二遍 loss 断崖），提出"哪些组件、怎么连"的假设，再用不同层级的证据一条条去验证：先看（注意力），再拆账（DLA），再动手改（消融、patching），最后直接读权重。每一条证据单独都不够，合在一起，才能说"我们理解了这个电路"，也才能预测它在没见过的输入上会怎么做。

### 想再玩一下？

- **自然文本里的感应**：把第 2 题改成 `cache["pattern", 1][0, 4]`，看 `1.4` 在句子第二次出现的 "Mrs Dursley" 上看哪里（提示：看 `D`、`urs` 两行）
- **换个上游头**：第 12 题把 `W_OV_07` 换成第 0 层别的头，对角线还在吗？
- **换一种关法**：第 8 题把置零改成换成平均值 `z[:, :, head, :] = z[:, :, head, :].mean(0)`，结果有什么不同？